In [ ]:
%pip install -U ultralytics


In [ ]:
!nvidia-smi


In [ ]:
import torch

torch.cuda.is_available()


In [ ]:
from ultralytics import solutions

inf = solutions.Inference(
    model="yolo11n.pt",  # you can use any model that Ultralytics supports, e.g., YOLO11, YOLOv10
)

inf.inference()

# Make sure to run the file using command `streamlit run path/to/file.py`


In [ ]:
!yolo solutions inference

!yolo solutions inference model="yolo26n.pt" # use model fine-tuned with Ultralytics Python package!


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)


In [ ]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture("path/to/video.mp4")
assert cap.isOpened(), "Error reading video file"

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("security_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

from_email = "abc@gmail.com"  # the sender email address
password = "---- ---- ---- ----"  # 16-digits password generated via: https://myaccount.google.com/apppasswords
to_email = "xyz@gmail.com"  # the receiver email address

# Initialize security alarm object
securityalarm = solutions.SecurityAlarm(
    show=True,  # display the output
    model="yolo26n.pt",  # e.g., yolo26s.pt, yolo26m.pt
    records=1,  # total detections count to send an email
)

securityalarm.authenticate(from_email, password, to_email)  # authenticate the email server

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = securityalarm(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)  # write the processed frame.

cap.release()
video_writer.release()
cv2.destroyAllWindows()  # destroy all opened windows


In [ ]:
### AI Security Servellience as AiSecurityGuard


In [ ]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture("../media_files/don_biye.jpg")
assert cap.isOpened(), "Error reading video file"

# Initialize object cropper
cropper = solutions.ObjectCropper(
    show=True,  # display the output
    model="yolo26m.pt",  # model for object cropping, e.g., yolo26x.pt.
    classes=[0, 2],  # crop specific classes such as person and car with the COCO pretrained model.
    conf=0.4,  # adjust confidence threshold for the objects.
    crop_dir="cropped-detections",  # set the directory name for cropped detections
)

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = cropper(im0)

    # print(results)  # access the output

cap.release()
cv2.destroyAllWindows()  # destroy all opened windows


In [ ]:
from ultralytics import YOLO
import cv2

# Load the recommended YOLO26 model
model = YOLO("yolo26n.pt")

results = model("../media_files/yolo check image and videos/download.jpg")  # replace with your image path

for r in results:
    for i, box in enumerate(r.boxes.xyxy):  # get box coordinates in (left, top, right, bottom)
        # Convert to integers for slicing
        x1, y1, x2, y2 = map(int, box)

        # Crop using NumPy slicing
        face_crop = r.orig_img[y1:y2, x1:x2]

        # Save the cropped face
        cv2.imwrite(f"face_{i}.jpg", face_crop)


# Base AI Security Alarm with Facial Recognition (YOLO + face_recognition) Template

In [ ]:
# Base AI Security Alarm with Facial Recognition (YOLO + face_recognition) Template

from ultralytics import solutions
from ultralytics.utils.plotting import Annotator

import os
import cv2
import numpy as np
import face_recognition
import pygame

# from ultralytics import solutions
from ultralytics import YOLO
from ultralytics.solutions.config import SolutionConfig
from ultralytics.utils import LOGGER

from ultralytics.solutions.solutions import BaseSolution, SolutionAnnotator, SolutionResults
from ultralytics.utils.plotting import colors

# ========== 🔊 SOUND SETUP ==========
pygame.mixer.init()
ALARM_FILE = "../media_files/Alarm-sound-samples/humordome-security-alert-sound-453297.mp3"
if os.path.exists(ALARM_FILE):
    pygame.mixer.music.load(ALARM_FILE)
else:
    print(f"[WARNING] Alarm file '{ALARM_FILE}' not found.")


# ========== 🧠 KNOWN FACE ENCODING LOADER ==========
KNOWN_FACE_DIR = "../family_members/"
known_face_encodings, known_face_names = [], []

if os.path.exists(KNOWN_FACE_DIR):
    for name in os.listdir(KNOWN_FACE_DIR):
        person_dir = os.path.join(KNOWN_FACE_DIR, name)
        if not os.path.isdir(person_dir):
            continue
        for filename in os.listdir(person_dir):
            path = os.path.join(person_dir, filename)
            try:
                img = face_recognition.load_image_file(path)
                enc = face_recognition.face_encodings(img)
                if enc:
                    known_face_encodings.append(enc[0])
                    known_face_names.append(name)
                    print(f"[INFO] Loaded face for {name} from {filename}")
            except Exception as e:
                print(f"[ERROR] Failed loading {path}: {e}")
else:
    print("[WARNING] No known_faces directory found.")


# ========== 👁️ FACE-RECOGNITION ALARM (REVISED & OPTIMIZED) ==========
class FaceRecognitionAlarmVisionEye(solutions.VisionEye):
    def __init__(self, *args, known_face_encodings=None, known_face_names=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.known_face_encodings = known_face_encodings or []
        self.known_face_names = known_face_names or []
        self.sound_played = False
        # Best practice: Set face recognition tolerance during initialization
        self.face_tolerance = 0.55
        self.vision_point = self.CFG["vision_point"]
        self.records = self.CFG.get("records", 1)
        # self.show = self.CFG.get("show", True)

    def play_sound(self):
        """Plays the alarm sound if it's not already playing."""
        if not self.sound_played:
            if pygame.mixer.get_init() and not pygame.mixer.music.get_busy():
                pygame.mixer.music.play()
                self.sound_played = True
                LOGGER.info("🚨 Alarm Triggered: Unknown person count reached threshold.")

    def reset_sound(self):
        """Stops the alarm sound and resets the state."""
        if self.sound_played:
            if pygame.mixer.get_init():
                pygame.mixer.music.stop()
            self.sound_played = False
            LOGGER.info("🟢 Alarm Reset: Area clear.")

    def __call__(self, im0):
        """
        Processes a single frame for person detection and face recognition.
        This implementation follows best practices for accuracy and performance.
        """
        # 1. Get person detections from the base class
        self.extract_tracks(im0)
        annotator = SolutionAnnotator(im0, line_width=self.line_width)

        unknown_person_count = 0

        # 2. Optimize by finding all faces in the frame at once (on a smaller version)
        # This is much faster than processing crops for each person.
        h, w, _ = im0.shape
        small_frame = cv2.resize(im0, (0, 0), fx=0.25, fy=0.25)
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

        # 3. Iterate through detected PERSONS from YOLO
        for box, conf, cls, t_id in zip(self.boxes, self.confs, self.clss, self.track_ids):
            if int(cls) == 0:  # Skip if not a person
                name = "Unknown"
                is_known = False

                # 4. Associate faces with person boxes
                # Check if any detected face is inside this person's bounding box
                person_box_left, person_box_top, person_box_right, person_box_bottom = map(int, box)

                for (face_top, face_right, face_bottom, face_left), face_encoding in zip(
                    face_locations, face_encodings
                ):
                    # Scale face locations back to original image size
                    face_top *= 4
                    face_right *= 4
                    face_bottom *= 4
                    face_left *= 4

                    # Check if the center of the face is inside the person's box
                    face_center_x = (face_left + face_right) // 2
                    face_center_y = (face_top + face_bottom) // 2

                    if (
                        person_box_left <= face_center_x <= person_box_right
                        and person_box_top <= face_center_y <= person_box_bottom
                    ):
                        # 5. Use robust face matching for the associated face
                        if self.known_face_encodings:
                            face_distances = face_recognition.face_distance(self.known_face_encodings, face_encoding)
                            best_match_index = np.argmin(face_distances)

                            if face_distances[best_match_index] < self.face_tolerance:
                                name = self.known_face_names[best_match_index]
                                is_known = True

                        # Once a face is matched to this person, stop checking other faces
                        break

                # 6. Update counter and draw labels
                if not is_known:
                    unknown_person_count += 1
                    color = (0, 0, 255)  # Red for Unknown
                    # label = f"Unknown ({conf:.2f})"
                    label = f"Unknown"
                else:
                    color = (0, 255, 0)  # Green for Known
                    label = f"{name}"
                    # label = f"{name} ({conf:.2f})"

                # annotator.box_label(box, label, color=color)

                # annotator.visioneye(box, self.vision_point)
                # build base label from the existing adjust_box_label()
                base_label = self.adjust_box_label(int(cls), float(conf) if conf is not None else 0.0, t_id)

                # custom label for 'person' class (COCO id 0). Use CFG override if provided.
                if int(cls) == 0:
                    prefix = str(self.CFG.get("person_label_prefix", label))
                    custom_label = f"{prefix}:"
                    # if base_label exists, concat both for full display
                    final_label = f"{custom_label} {base_label}" if base_label else custom_label
                else:
                    final_label = base_label

                # draw final label and vision eye mapping
                annotator.box_label(box, label=final_label, color=colors(int(t_id), True))
            else:
                # For non-person classes, use default labeling
                annotator.box_label(box, label=self.adjust_box_label(cls, conf, t_id), color=colors(int(t_id), True))

            annotator.visioneye(box, self.vision_point)

        # 7. Trigger alarm based on the COUNT of unknown people and the 'records' threshold
        if unknown_person_count >= self.records:
            self.play_sound()
        else:
            self.reset_sound()

        plot_im = annotator.result()
        self.display_output(plot_im)

        # Display track count on the frame
        total_tracks = len(getattr(self, "track_ids", []))
        cv2.putText(plot_im, f"Tracks: {total_tracks}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        return SolutionResults(plot_im=plot_im, total_tracks=len(self.track_ids))


if __name__ == "__main__":
    # cap = cv2.VideoCapture(0)
    # cap = cv2.VideoCapture("../media_files/WIN_20260321_14_46_59_Pro.mp4")
    # cap = cv2.VideoCapture("../media_files/WIN_20260228_13_25_34_Pro.mp4")
    # cap = cv2.VideoCapture("../media_files/WIN_20260227_22_00_29_Pro.mp4")
    # cap = cv2.VideoCapture("media_files/person/ruhama/VID_20251122_142652.mp4")
    # cap = cv2.VideoCapture("../media_files/istockphoto-2205327227-640_adpp_is.mp4")
    cap = cv2.VideoCapture("../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4")
    # cap = cv2.VideoCapture("../media_files/istockphoto-2205327227-640_adpp_is.mp4")
    # cap = cv2.VideoCapture("../media_files/Logi C270 HD WebCam 2025-10-29 22-29-17.mp4")
    # cap = cv2.VideoCapture("../media_files/ruhama.mp4")
    # cap = cv2.VideoCapture("../media_files/ruhama.mp4")
    # assert cap.isOpened(), "Error reading video file"

    # Video writer
    w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
    video_writer = cv2.VideoWriter("visioneye_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    # Initialize vision eye object
    visioneyeInterface = FaceRecognitionAlarmVisionEye(
        show=True,  # display the output
        model="yolo26m.pt",  # use any model that Ultralytics support, i.e, YOLOv10
        # classes=[0, 19],  # generate visioneye view for specific classes
        # vision_point=(850, 550),  # the point, where vision will view objects and draw tracks
        known_face_encodings=known_face_encodings,
        known_face_names=known_face_names,
        records=1,  # number of unknown persons to trigger alqqarm
        conf=0.1,
        iou=0.9,
        verbose=True,
        # show_labels=True,
    )
    # print(visioneyeInterface)

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = visioneyeInterface(im0)

    print(results)  # access the output

    video_writer.write(results.plot_im)  # write the video file

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()


In [ ]:
%pip install deepface


In [ ]:
import cv2
from ultralytics import YOLO
from deepface import DeepFace

# 1. Load Ultralytics YOLO26 for face detection
model = YOLO("yolo26n.pt")

# 2. Run inference
img_path = "../family_members/robin/robin2.jpg"  # path to your image
results = model(img_path)

# 3. Process detections
for result in results:
    for box in result.boxes.xyxy:
        # Convert tensor coordinates to integers
        x1, y1, x2, y2 = map(int, box)

        # Crop the face from the original image
        face_crop = result.orig_img[y1:y2, x1:x2]

        # 4. Perform recognition or analysis with DeepFace
        objs = DeepFace.analyze(face_crop, actions=["age", "gender", "race"])
        print(objs)


In [ ]:
from ppadb.client import Client as AdbClient

# Connect to ADB server
client = AdbClient(host="127.0.0.1", port=5037)
devices = client.devices()
device = devices[0]

# Make the call
phone_number = "1234567890"
device.shell(f"am start -a android.intent.action.CALL -d tel:{phone_number}")


In [ ]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture("../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4")
assert cap.isOpened(), "Error reading video file"

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("security_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

from_email = "abc@gmail.com"  # the sender email address
password = "---- ---- ---- ----"  # 16-digits password generated via: https://myaccount.google.com/apppasswords
to_email = "xyz@gmail.com"  # the receiver email address

# Initialize security alarm object
securityalarm = solutions.SecurityAlarm(
    show=True,  # display the output
    model="yolo26m.pt",  # e.g., yolo26s.pt, yolo26m.pt
    records=1,  # total detections count to send an email
)

# securityalarm.authenticate(from_email, password, to_email)  # authenticate the email server

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = securityalarm(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)  # write the processed frame.

cap.release()
video_writer.release()
cv2.destroyAllWindows()  # destroy all opened windows


In [3]:
import mediapipe as mp
import cv2
import numpy as np
import os
from scipy.spatial import distance as dist

# Configuration
THRESHOLD_MIN = 80
THRESHOLD_MAX = 200
SAVE_DIR = "cropped_faces"
os.makedirs(SAVE_DIR, exist_ok=True)

# Initialize MediaPipe face detector
mp_face_detection = mp.solutions.face_detection
face_detector = mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.6)


def get_quality_score(image):
    """Calculates a quality score based on sharpness (Laplacian) and exposure."""
    sharpness = cv2.Laplacian(image, cv2.CV_64F).var()
    yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    brightness = np.mean(yuv[:, :, 0])
    exposure_score = 255 - abs(brightness - 128)
    return (sharpness * exposure_score), brightness


# cap = cv2.VideoCapture("../media_files/istockphoto-2002566174-640_adpp_is.mp4")
# cap = cv2.VideoCapture("../media_files/istockphoto-2152802033-640_adpp_is.mp4")
# cap = cv2.VideoCapture("../media_files/istockphoto-2002563994-640_adpp_is.mp4")
# cap = cv2.VideoCapture("../media_files/DSC_0098_edited.jpg")
# cap = cv2.VideoCapture("../media_files/istockphoto-2240284006-640_adpp_is.mp4")
# cap = cv2.VideoCapture("../media_files/istockphoto-2240284006-640_adpp_is.mp4")
cap = cv2.VideoCapture("../media_files/WIN_20260228_13_25_34_Pro.mp4")
if not cap.isOpened():
    raise RuntimeError("Cannot open video file.")

# Tracking state
next_id = 0
trackers = {}  # {track_id: centroid}
best_shots = {}  # {track_id: {'score': float, 'crop': image, 'brightness': float}}

print("Processing video to extract the best unique face for each person...")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    ih, iw, _ = frame.shape
    results = face_detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    current_frame_faces = []
    if results.detections:
        for detection in results.detections:
            bbox = detection.location_data.relative_bounding_box
            x, y = max(0, int(bbox.xmin * iw)), max(0, int(bbox.ymin * ih))
            w, h = max(1, int(bbox.width * iw)), max(1, int(bbox.height * ih))
            cx, cy = x + w // 2, y + h // 2
            current_frame_faces.append({"box": (x, y, w, h), "centroid": (cx, cy)})

    # Match detected faces to persistent IDs using Centroid Tracking
    if current_frame_faces:
        if not trackers:
            for f in current_frame_faces:
                trackers[next_id] = f["centroid"]
                next_id += 1
        else:
            ids = list(trackers.keys())
            coords = list(trackers.values())

            for f in current_frame_faces:
                # Find closest existing person
                distances = dist.cdist([f["centroid"]], coords)
                idx = np.argmin(distances)

                if distances[0][idx] < 115:  # Threshold for 'same person'
                    tid = ids[idx]
                    trackers[tid] = f["centroid"]

                    # Evaluate quality for current track
                    x, y, w, h = f["box"]
                    face_roi = frame[y : y + h, x : x + w]
                    if face_roi.size > 0:
                        score, brightness = get_quality_score(face_roi)
                        if THRESHOLD_MIN < brightness < THRESHOLD_MAX:
                            # Update best shot if current quality score is higher
                            if tid not in best_shots or score > best_shots[tid]["score"]:
                                pw, ph = int(w * 0.2), int(h * 0.2)
                                crop = frame[max(0, y - ph) : min(ih, y + h + ph), max(0, x - pw) : min(iw, x + w + pw)]
                                if crop.size > 0:
                                    best_shots[tid] = {"score": score, "crop": crop.copy(), "brightness": brightness}
                else:
                    # New person detected
                    trackers[next_id] = f["centroid"]
                    next_id += 1

    cv2.imshow("Best Face Hunter", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

# Save final unique best faces
print(f"\n✅ Video processed. Found {len(best_shots)} unique persons.")
for tid, data in best_shots.items():
    filename = os.path.join(SAVE_DIR, f"person_{tid}_best.jpg")
    cv2.imwrite(filename, data["crop"])
    print(f"✨ Saved: {filename} (Brightness: {data['brightness']:.1f}, Quality Score: {data['score']:.1f})")


Processing video to extract the best unique face for each person...

✅ Video processed. Found 2 unique persons.
✨ Saved: cropped_faces\person_0_best.jpg (Brightness: 95.2, Quality Score: 32695.3)
✨ Saved: cropped_faces\person_2_best.jpg (Brightness: 164.7, Quality Score: 31203.8)


In [ ]:
"""
Security Alarm System with Face Recognition and YOLO Person Detection

Features:
- Real-time pe File rson detection using YOLO
- Face recognition against known faces database
- Best-quality face capture for each tracked person
- Alarm triggering for unknown persons

Author: Security AI Team
Version: 2.0.0
"""

from __future__ import annotations

import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import pygame
from scipy.spatial import distance as dist

from ultralytics import solutions
from ultralytics.solutions.solutions import SolutionAnnotator, SolutionResults
from ultralytics.utils import LOGGER
import face_recognition


# =============================================================================
# CONFIGURATION
# =============================================================================


@dataclass
class SecurityConfig:
    """Centralized configuration for the security system."""

    # Directories
    save_dir: str = "person_cropped_face"
    known_faces_dir: str = "../family_members/"
    alarm_file: str = "../media_files/Alarm-sound-samples/humordome-security-alert-sound-453297.mp3"

    # Quality thresholds
    sharpness_threshold: int = 40
    brightness_min: int = 80
    brightness_max: int = 200
    quality_weight_exposure: float = 1.0
    quality_weight_sharpness: float = 0.5

    # Face recognition
    face_tolerance: float = 0.55

    # Performance
    face_detection_scale: float = 0.25
    frame_skip_interval: int = 2

    # Alarm
    unknown_threshold: int = 1

    # Tracking
    centroid_distance_threshold: float = 100.0
    padding_ratio: float = 0.2

    # Model
    yolo_model: str = "yolo11n.pt"
    confidence: float = 0.5
    iou: float = 0.7

    def __post_init__(self):
        """Validate configuration after initialization."""
        os.makedirs(self.save_dir, exist_ok=True)

        if not Path(self.alarm_file).exists():
            LOGGER.warning(f"⚠️ Alarm file not found: {self.alarm_file}")


# =============================================================================
# QUALITY ASSESSMENT
# =============================================================================


class FaceQualityAnalyzer:
    """Analyzes face image quality for best shot selection."""

    def __init__(self, config: SecurityConfig):
        self.config = config

    def calculate_score(self, face_img: np.ndarray) -> tuple[float, float]:
        """Calculate quality score based on sharpness and exposure."""
        if face_img.size == 0:
            return 0.0, 0.0

        # Sharpness Score (Laplacian variance)
        sharpness = cv2.Laplacian(face_img, cv2.CV_64F).var()

        # Exposure Score
        yuv = cv2.cvtColor(face_img, cv2.COLOR_BGR2YUV)
        brightness = np.mean(yuv[:, :, 0])
        exposure_score = 255 - abs(brightness - 128)

        # Check brightness bounds
        if not (self.config.brightness_min <= brightness <= self.config.brightness_max):
            return 0.0, float(brightness)

        # Discard too blurry images
        if sharpness < self.config.sharpness_threshold:
            return 0.0, float(brightness)

        quality_score = (
            exposure_score * self.config.quality_weight_exposure + sharpness * self.config.quality_weight_sharpness
        )

        return float(quality_score), float(brightness)


# =============================================================================
# FACE DATABASE
# =============================================================================


class FaceDatabase:
    """Manages known face encodings and names."""

    def __init__(self, known_faces_dir: str):
        self.known_faces_dir = Path(known_faces_dir)
        self.encodings: list[np.ndarray] = []
        self.names: list[str] = []
        self._load_known_faces()

    def _load_known_faces(self) -> None:
        """Load and encode known faces from directory."""
        if not self.known_faces_dir.exists():
            LOGGER.warning(f"⚠️ Known faces directory not found: {self.known_faces_dir}")
            return

        for person_name in os.listdir(self.known_faces_dir):
            person_path = self.known_faces_dir / person_name

            if not person_path.is_dir():
                continue

            for image_file in person_path.iterdir():
                if image_file.suffix.lower() not in (".jpg", ".jpeg", ".png", ".bmp"):
                    continue

                try:
                    image = face_recognition.load_image_file(str(image_file))
                    encodings = face_recognition.face_encodings(image)

                    if encodings:
                        self.encodings.append(encodings[0])
                        self.names.append(person_name)
                        LOGGER.info(f"✅ Loaded face for '{person_name}' from {image_file.name}")
                    else:
                        LOGGER.warning(f"⚠️ No face detected in {image_file}")

                except Exception as e:
                    LOGGER.error(f"❌ Failed to load {image_file}: {e}")

        LOGGER.info(f"📊 Loaded {len(self.encodings)} known faces")


# =============================================================================
# TRACKING SYSTEM
# =============================================================================


class PersonTracker:
    """Tracks persons across frames using centroid tracking."""

    def __init__(self, config: SecurityConfig):
        self.config = config
        self.next_id: int = 0
        self.centroids: dict[int, tuple[int, int]] = {}
        self.best_shots: dict[int, dict] = {}

    def update(self, detections: list[dict]) -> dict[int, np.ndarray]:
        """Update tracking with new detections."""
        if not detections:
            self.centroids.clear()
            return {}

        current_centroids = [d["centroid"] for d in detections]

        # First frame - initialize
        if not self.centroids:
            for det in detections:
                self.centroids[self.next_id] = det["centroid"]
                self.best_shots[self.next_id] = {"score": 0.0, "crop": None, "brightness": 0.0, "box": det["box"]}
                self.next_id += 1
            return {i: self.centroids[i] for i in range(len(detections))}

        # Match existing tracks to new detections
        tracked_boxes: dict[int, np.ndarray] = {}
        used_detections = set()

        track_ids = list(self.centroids.keys())
        existing_centroids = list(self.centroids.values())

        for det_idx, det in enumerate(detections):
            distances = dist.cdist([det["centroid"]], existing_centroids)[0]
            min_idx = np.argmin(distances)

            if distances[min_idx] < self.config.centroid_distance_threshold:
                track_id = track_ids[min_idx]
                self.centroids[track_id] = det["centroid"]
                tracked_boxes[track_id] = np.array(det["box"])
                used_detections.add(det_idx)

        # Handle new detections
        for det_idx, det in enumerate(detections):
            if det_idx not in used_detections:
                new_id = self.next_id
                self.next_id += 1
                self.centroids[new_id] = det["centroid"]
                self.best_shots[new_id] = {"score": 0.0, "crop": None, "brightness": 0.0, "box": det["box"]}
                tracked_boxes[new_id] = np.array(det["box"])

        return tracked_boxes

    def update_best_shot(self, track_id: int, crop: np.ndarray, score: float, brightness: float) -> bool:
        """Update best shot if quality is better."""
        if track_id not in self.best_shots:
            return False

        current_best = self.best_shots[track_id]

        if score > current_best["score"]:
            self.best_shots[track_id] = {
                "score": score,
                "crop": crop.copy(),
                "brightness": brightness,
                "box": current_best["box"],
            }
            return True
        return False

    def save_best_shots(self, save_dir: str) -> dict[int, str]:
        """Save all best shots to disk."""
        saved_paths = {}

        for track_id, data in self.best_shots.items():
            if data["crop"] is not None:
                filename = f"person_{track_id}_best.jpg"
                filepath = Path(save_dir) / filename
                cv2.imwrite(str(filepath), data["crop"])
                saved_paths[track_id] = str(filepath)
                LOGGER.info(f"💾 Saved best shot for track {track_id}")

        return saved_paths


# =============================================================================
# ALARM SYSTEM
# =============================================================================


class AlarmSystem:
    """Manages alarm sound playback."""

    def __init__(self, alarm_file: str):
        self.alarm_file = Path(alarm_file)
        self.is_playing: bool = False
        self._initialize_mixer()

    def _initialize_mixer(self) -> None:
        """Initialize pygame mixer with error handling."""
        try:
            if not pygame.mixer.get_init():
                pygame.mixer.init(frequency=44100, size=-16, channels=2, buffer=512)

            if self.alarm_file.exists():
                pygame.mixer.music.load(str(self.alarm_file))
                LOGGER.info(f"✅ Loaded alarm sound: {self.alarm_file}")
            else:
                LOGGER.warning(f"⚠️ Alarm file not found: {self.alarm_file}")

        except pygame.error as e:
            LOGGER.error(f"❌ Failed to initialize pygame mixer: {e}")

    def play(self) -> None:
        """Play alarm sound if not already playing."""
        if not self.is_playing and pygame.mixer.get_init():
            try:
                if not pygame.mixer.music.get_busy():
                    pygame.mixer.music.play(-1)
                    self.is_playing = True
                    LOGGER.info("🚨 ALARM TRIGGERED")
            except pygame.error as e:
                LOGGER.error(f"❌ Failed to play alarm: {e}")

    def stop(self) -> None:
        """Stop alarm sound."""
        if self.is_playing and pygame.mixer.get_init():
            try:
                pygame.mixer.music.stop()
                self.is_playing = False
            except pygame.error as e:
                LOGGER.error(f"❌ Failed to stop alarm: {e}")


# =============================================================================
# MAIN SECURITY SYSTEM
# =============================================================================


class SecurityVisionEye(solutions.VisionEye):
    """
    Main security system combining YOLO person detection with face recognition.
    """

    def __init__(
        self,
        *args,
        config: Optional[SecurityConfig] = None,
        known_face_encodings: Optional[list] = None,
        known_face_names: Optional[list] = None,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.config = config or SecurityConfig()
        self.known_face_encodings = known_face_encodings or []
        self.known_face_names = known_face_names or []

        self.quality_analyzer = FaceQualityAnalyzer(self.config)
        self.tracker = PersonTracker(self.config)
        self.alarm = AlarmSystem(self.config.alarm_file)

        self.frame_count: int = 0
        self._face_cache: dict = {}

    def _detect_faces(self, frame: np.ndarray) -> list[tuple]:
        """Detect faces in frame with optimization."""
        h, w = frame.shape[:2]
        small_frame = cv2.resize(
            frame, (0, 0), fx=self.config.face_detection_scale, fy=self.config.face_detection_scale
        )
        rgb_small = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

        face_locations = face_recognition.face_locations(rgb_small)

        scale_factor = 1 / self.config.face_detection_scale
        scaled_locations = []
        for top, right, bottom, left in face_locations:
            scaled_locations.append(
                (
                    int(top * scale_factor),
                    int(right * scale_factor),
                    int(bottom * scale_factor),
                    int(left * scale_factor),
                )
            )

        return scaled_locations

    def _identify_face(self, face_encoding: np.ndarray) -> tuple[str, bool]:
        """Identify a face against known encodings."""
        if not self.known_face_encodings:
            return "Unknown", False

        distances = face_recognition.face_distance(self.known_face_encodings, face_encoding)

        if len(distances) == 0:
            return "Unknown", False

        best_idx = np.argmin(distances)

        if distances[best_idx] < self.config.face_tolerance:
            return self.known_face_names[best_idx], True

        return "Unknown", False

    def _crop_face_with_padding(self, frame: np.ndarray, box: tuple, padding_ratio: float = 0.2) -> np.ndarray:
        """Crop face region with padding."""
        x1, y1, x2, y2 = box
        h, w = frame.shape[:2]

        pw = int((x2 - x1) * padding_ratio)
        ph = int((y2 - y1) * padding_ratio)

        x1_pad = max(0, x1 - pw)
        y1_pad = max(0, y1 - ph)
        x2_pad = min(w, x2 + pw)
        y2_pad = min(h, y2 + ph)

        return frame[y1_pad:y2_pad, x1_pad:x2_pad]

    def __call__(self, im0: np.ndarray) -> SolutionResults:
        """Process a single frame."""
        self.frame_count += 1

        self.extract_tracks(im0)
        annotator = SolutionAnnotator(im0, line_width=self.line_width)

        # Detect faces with frame skipping optimization
        face_locations = []
        if self.frame_count % self.config.frame_skip_interval == 0:
            face_locations = self._detect_faces(im0)
            rgb_small = cv2.cvtColor(
                cv2.resize(im0, (0, 0), fx=self.config.face_detection_scale, fy=self.config.face_detection_scale),
                cv2.COLOR_BGR2RGB,
            )
            face_encodings = face_recognition.face_encodings(rgb_small, face_locations) if face_locations else []

            # Rebuild cache
            self._face_cache = {}
            scale_factor = 1 / self.config.face_detection_scale
            for (top, right, bottom, left), enc in zip(face_locations, face_encodings):
                top_s = int(top * scale_factor)
                right_s = int(right * scale_factor)
                bottom_s = int(bottom * scale_factor)
                left_s = int(left * scale_factor)
                cx, cy = (left_s + right_s) // 2, (top_s + bottom_s) // 2
                self._face_cache[(cx, cy)] = (enc, (top_s, right_s, bottom_s, left_s))

        unknown_count = 0
        person_detections = []

        for box, conf, cls, track_id in zip(self.boxes, self.confs, self.clss, self.track_ids):
            if int(cls) != 0:
                continue

            box = box.astype(int)
            px1, py1, px2, py2 = box

            # Find associated face
            person_face_encoding = None
            face_box = None

            for (cx, cy), (enc, fbox) in self._face_cache.items():
                if px1 <= cx <= px2 and py1 <= cy <= py2:
                    person_face_encoding = enc
                    face_box = fbox
                    break

            # Identify person
            name = "Unknown"
            is_known = False

            if person_face_encoding is not None:
                name, is_known = self._identify_face(person_face_encoding)

                # Quality assessment and best shot capture
                if face_box is not None:
                    face_crop = im0[face_box[0] : face_box[2], face_box[1] : face_box[3]]
                    quality_score, brightness = self.quality_analyzer.calculate_score(face_crop)

                    if quality_score > 0:
                        padded_crop = self._crop_face_with_padding(im0, face_box, self.config.padding_ratio)
                        self.tracker.update_best_shot(track_id, padded_crop, quality_score, brightness)

            # Track for centroid tracking
            cx, cy = (px1 + px2) // 2, (py1 + py2) // 2
            person_detections.append({"box": (px1, py1, px2, py2), "centroid": (cx, cy)})

            if not is_known:
                unknown_count += 1

            # Annotate
            label = f"{name} ID:{track_id}"
            color = (0, 255, 0) if is_known else (0, 0, 255)
            annotator.box_label(box, label, color=color)
            annotator.visioneye(box, self.vision_point)

        # Update tracker
        self.tracker.update(person_detections)

        # Alarm logic
        if unknown_count >= self.config.unknown_threshold:
            self.alarm.play()
        else:
            self.alarm.stop()

        # Generate output
        plot_im = annotator.result()
        self.display_output(plot_im)

        cv2.putText(
            plot_im, f"Tracks: {len(self.track_ids)}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2
        )

        return SolutionResults(plot_im=plot_im, total_tracks=len(self.track_ids))


# =============================================================================
# MAIN ENTRY POINT
# =============================================================================


def main():
    """Main function to run the security system."""

    config = SecurityConfig(
        yolo_model="yolo11n.pt",
        save_dir="person_cropped_face",
        known_faces_dir="../family_members/",
        alarm_file="../media_files/Alarm-sound-samples/humordome-security-alert-sound-453297.mp3",
        unknown_threshold=1,
        frame_skip_interval=2,
        confidence=0.5,
        iou=0.7,
    )

    # Load known faces
    face_db = FaceDatabase(config.known_faces_dir)

    # Video source
    video_path = "../media_files/istockphoto-2002566174-640_adpp_is.mp4"

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        LOGGER.error(f"❌ Cannot open video: {video_path}")
        sys.exit(1)

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    output_path = "security_output.mp4"
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    security = SecurityVisionEye(
        show=True,
        model=config.yolo_model,
        vision_point=(w // 2, h // 2),
        known_face_encodings=face_db.encodings,
        known_face_names=face_db.names,
        records=config.unknown_threshold,
        conf=config.confidence,
        iou=config.iou,
        verbose=True,
    )

    try:
        while cap.isOpened():
            success, frame = cap.read()

            if not success:
                LOGGER.info("✅ Video processing complete")
                break

            results = security(frame)
            writer.write(results.plot_im)

            cv2.imshow("Security Vision", results.plot_im)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    finally:
        cap.release()
        writer.release()
        cv2.destroyAllWindows()

        security.tracker.save_best_shots(config.save_dir)

        LOGGER.info(f"💾 Output saved to: {output_path}")


if __name__ == "__main__":
    main()


In [ ]:
import cv2
import threading
import queue
from ultralytics import YOLO
from deepface import DeepFace


class SecuritySystem:
    def __init__(self, model_path="yolo26n.pt", db_path=".known_faces/"):
        # 1. Initialize YOLOv26 (Detection + Tracking)
        self.detector = YOLO(model_path)

        # 2. Recognition Queue & State
        self.face_queue = queue.Queue(maxsize=5)
        self.known_db = db_path
        self.tracked_identities = {}  # {track_id: {"name": "Unknown", "verified": False}}

        # 3. Start Background Recognition Thread
        self.recognition_thread = threading.Thread(target=self._recognition_worker, daemon=True)
        self.recognition_thread.start()

    def _recognition_worker(self):
        """Asynchronous worker to prevent UI lag during DeepFace inference."""
        while True:
            face_img, track_id = self.face_queue.get()
            try:
                # DeepFace.find is more efficient than .verify for databases
                results = DeepFace.find(img_path=face_img, db_path=self.known_db, enforce_detection=False, silent=True)

                if len(results) > 0 and not results[0].empty:
                    name = results[0].iloc[0]["identity"].split("/")[-2]  # Folder name as ID
                    self.tracked_identities[track_id] = {"name": name, "verified": True}
                else:
                    self.tracked_identities[track_id] = {"name": "UNKNOWN", "verified": True}
                    self._trigger_alarm(f"Unknown person (ID:{track_id}) detected!")
            except Exception as e:
                print(f"Recognition Error: {e}")
            self.face_queue.task_done()

    def _trigger_alarm(self, message):
        print(f"🚨 ALARM: {message}")  # Connect to GPIO/Sound API here

    def run(self, source="../media_files/istockphoto-2240284006-640_adpp_is.mp4"):
        cap = cv2.VideoCapture(source)

        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            # 4. Perform YOLO Tracking (Using ByteTrack or BoT-SORT)
            results = self.detector.track(frame, persist=True, classes=[0])  # Class 0 = Person

            if results[0].boxes.id is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                track_ids = results[0].boxes.id.cpu().numpy().astype(int)

                for box, track_id in zip(boxes, track_ids):
                    # Only process recognition if we haven't verified this ID yet
                    if track_id not in self.tracked_identities:
                        x1, y1, x2, y2 = map(int, box)
                        face_crop = frame[y1:y2, x1:x2]

                        if not self.face_queue.full():
                            self.face_queue.put((face_crop, track_id))
                            self.tracked_identities[track_id] = {"name": "Processing...", "verified": False}

                    # 5. Draw Visual Feedback
                    label = self.tracked_identities.get(track_id, {}).get("name", "New")
                    cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 255, 0), 2)
                    cv2.putText(
                        frame,
                        f"ID:{track_id} {label}",
                        (int(box[0]), int(box[1] - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (255, 255, 255),
                        2,
                    )

            cv2.imshow("Security AI Inference", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

        cap.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    system = SecuritySystem()
    system.run()


In [ ]:
import cv2
import os
import threading
from ultralytics import YOLO, solutions
from deepface import DeepFace


class AISecurityGuard:
    def __init__(self):
        # 1. Models: Using 'yolo11n-pose.pt' (corrected from yolo26n)
        self.model = YOLO("yolo11n-pose.pt")
        self.face_db = "family_members"

        # 2. Solutions: VisionEye for spatial mapping
        # FIX: Replaced invalid 'view_img' with 'show'
        # Note: VisionEye may require a 'vision_point' arg depending on configuration
        self.eye = solutions.VisionEye(show=False)

        # Note: 'solutions.DistanceCalculation' is not a standard Ultralytics module in the provided context.
        # Ensure you have this class defined or imported. If using SpeedEstimator or similar, adjust accordingly.
        # self.dist_tool = solutions.DistanceCalculation(model="yolo11n.pt")

        # 3. State Management
        self.verified_tracks = {}  # {track_id: "Name"}
        self.cattle_ids = [19, 20]  # COCO classes for cow/sheep

    def process_face(self, frame, box, track_id):
        """Runs DeepFace in a non-blocking background thread."""
        x1, y1, x2, y2 = map(int, box)

        # Ensure coordinates are within frame bounds
        h, w = frame.shape[:2]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)

        if x2 <= x1 or y2 <= y1:
            return

        face_crop = frame[y1:y2, x1:x2]

        # Ensure directory exists
        os.makedirs("captured_faces", exist_ok=True)
        # Save 'best' crop for logging as requested
        cv2.imwrite(f"captured_faces/track_{track_id}.jpg", face_crop)

        try:
            # Find identity in family_members folder
            results = DeepFace.find(face_crop, db_path=self.face_db, silent=True)
            if results and not results[0].empty:
                name = results[0].iloc[0]["identity"].split(os.sep)[-2]
                self.verified_tracks[track_id] = name
            else:
                self.verified_tracks[track_id] = "UNAUTHORIZED"
                self.trigger_alarm(f"Intruder Detected: ID {track_id}")
        except Exception as e:
            # print(f"Face processing error: {e}")
            self.verified_tracks[track_id] = "UNKNOWN"

    def monitor_farm(self, source="../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4"):
        cap = cv2.VideoCapture(source)

        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            # Detection & Tracking
            results = self.model.track(frame, persist=True, tracker="botsort.yaml")

            if results[0].boxes.id is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                track_ids = results[0].boxes.id.cpu().numpy().astype(int)
                cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)

                # Logic for Cattle vs Person
                for box, tid, cid in zip(boxes, track_ids, cls_ids):
                    # A. Recognition Logic (only run once per ID)
                    if cid == 0 and tid not in self.verified_tracks:
                        # Pass a copy of the frame to the thread to be safe
                        threading.Thread(target=self.process_face, args=(frame.copy(), box, tid)).start()

                    # B. Anomaly/Pose Detection (e.g., person crouching near cattle)
                    if cid == 0 and results[0].keypoints is not None:
                        keypoints = results[0].keypoints.xy[0].cpu().numpy()
                        # Ensure we have enough keypoints and they are valid (conf > 0)
                        if len(keypoints) > 11 and keypoints[0][1] > 0 and keypoints[11][1] > 0:
                            # Simple senior-level logic: If head (idx 0) is lower (higher Y) than hips (idx 11)
                            if keypoints[0][1] > keypoints[11][1]:
                                self.trigger_alarm("Suspicious Posing (Crouching) detected!")

                # C. Distance Calculation (Theft Prevention)
                # Ensure self.dist_tool is initialized and valid before calling
                # if hasattr(self, 'dist_tool'):
                #     frame = self.dist_tool.calculate_distance(frame, results)

            cv2.imshow("AI Night Guard", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

        cap.release()
        cv2.destroyAllWindows()

    def trigger_alarm(self, msg):
        print(f"ALARM: {msg}")
        # Logic for Email/Siren goes here


if __name__ == "__main__":
    guard = AISecurityGuard()
    guard.monitor_farm()
